# Flight Tracker Pipeline — Data Exploration

This notebook pulls live aircraft position data from the OpenSky Network API and loads it into Postgres.

**Structure:**
1. Imports & configuration
2. Database connection (run once per session)
3. Create table (run once ever — safe to re-run, uses `IF NOT EXISTS`)
4. Fetch + load pipeline (run this repeatedly to ingest new snapshots)
5. Verification queries

## 1. Imports & configuration

In [1]:
import requests
import pandas as pd
from sqlalchemy import create_engine, text

OPENSKY_URL = "https://opensky-network.org/api/states/all"

DB_HOST = "postgres"       # service name from docker-compose.yml, not localhost
DB_PORT = 5432
DB_NAME = "mydatabase"
DB_USER = "root"
DB_PASSWORD = "root"

# Raw column order returned by the OpenSky /states/all endpoint
OPENSKY_COLUMNS = [
    "icao24", "callsign", "origin_country", "time_position",
    "last_contact", "longitude", "latitude", "baro_altitude",
    "on_ground", "velocity", "true_track", "vertical_rate",
    "sensors", "geo_altitude", "squawk", "spi", "position_source"
]

## 2. Database connection

Run this once per session — every other cell in this notebook reuses `engine`.

In [2]:
engine = create_engine(
    f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Quick connectivity check
with engine.connect() as conn:
    version = conn.execute(text("SELECT version();")).fetchone()
print(version[0])

PostgreSQL 18.4 (Debian 18.4-1.pgdg13+1) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit


## 3. Create table (one-time setup)

Safe to re-run any time — `IF NOT EXISTS` means it won't touch the table if it's already there.

In [3]:
CREATE_TABLE_SQL = """
CREATE TABLE IF NOT EXISTS flight_positions (
    id SERIAL PRIMARY KEY,
    icao24 VARCHAR(10),
    callsign VARCHAR(20),
    origin_country VARCHAR(100),
    time_position BIGINT,
    last_contact BIGINT,
    longitude DOUBLE PRECISION,
    latitude DOUBLE PRECISION,
    baro_altitude DOUBLE PRECISION,
    on_ground BOOLEAN,
    velocity DOUBLE PRECISION,
    true_track DOUBLE PRECISION,
    vertical_rate DOUBLE PRECISION,
    geo_altitude DOUBLE PRECISION,
    squawk VARCHAR(10),
    spi BOOLEAN,
    position_source INTEGER,
    ingested_at TIMESTAMP DEFAULT NOW()
);
"""

with engine.connect() as conn:
    conn.execute(text(CREATE_TABLE_SQL))
    conn.commit()

print("Table ready.")

Table ready.


## 4. Fetch + load pipeline

This is the part you re-run every time you want a fresh snapshot of live flights.

In [4]:
def fetch_states() -> pd.DataFrame:
    """Pull the current global aircraft state vector from OpenSky."""
    response = requests.get(OPENSKY_URL)
    response.raise_for_status()
    data = response.json()

    df = pd.DataFrame(data["states"], columns=OPENSKY_COLUMNS)
    df["callsign"] = df["callsign"].str.strip()
    df = df.drop(columns=["sensors"])  # always None — not in the table schema

    print(f"Fetched {len(df)} aircraft at {pd.Timestamp.now()}")
    return df


import math

def load_states(df: pd.DataFrame) -> None:
    """Append a snapshot of aircraft states into Postgres, skipping exact duplicates."""
    records = df.to_dict(orient="records")

    # Replace NaN with None in each record — dicts can hold real None, unlike float64 columns
    for row in records:
        for key, value in row.items():
            if isinstance(value, float) and math.isnan(value):
                row[key] = None

    insert_sql = text("""
        INSERT INTO flight_positions
            (icao24, callsign, origin_country, time_position, last_contact,
             longitude, latitude, baro_altitude, on_ground, velocity,
             true_track, vertical_rate, geo_altitude, squawk, spi, position_source)
        VALUES
            (:icao24, :callsign, :origin_country, :time_position, :last_contact,
             :longitude, :latitude, :baro_altitude, :on_ground, :velocity,
             :true_track, :vertical_rate, :geo_altitude, :squawk, :spi, :position_source)
        ON CONFLICT (icao24, time_position) DO NOTHING
    """)

    with engine.connect() as conn:
        conn.execute(insert_sql, records)
        conn.commit()

    print(f"Attempted to load {len(df)} rows (duplicates skipped automatically).")

In [5]:
df = fetch_states()
load_states(df)
df.head()

Fetched 9001 aircraft at 2026-08-07 00:34:41.971045
Attempted to load 9001 rows (duplicates skipped automatically).


,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source
0,e8027d,LPE2051,Chile,1.786063e+09,1786062878,-79.8610,-6.6377,11277.60,False,230.59,152.06,0.33,12039.60,None,False,0
1,c81bd2,ZKJMR,New Zealand,1.786063e+09,1786062877,176.7774,-39.2334,1988.82,False,58.79,344.26,0.00,2118.36,None,False,0
2,aa9321,UAL915,United States,1.786063e+09,1786062878,-62.1834,49.5611,11277.60,False,272.07,52.22,0.00,11871.96,2312,False,0
3,801640,AIC2385,India,1.786063e+09,1786062877,95.5716,10.8261,10363.20,False,235.20,279.44,0.00,NaN,2126,False,0
4,801645,AIC7JN,India,1.786063e+09,1786062878,77.6974,13.5808,4968.24,False,171.73,15.29,8.13,5143.50,None,False,0


In [10]:
import time

def poll_forever(interval_seconds: int = 30):
    """Continuously fetch and load OpenSky data every `interval_seconds`."""
    while True:
        try:
            df = fetch_states()
            load_states(df)
        except Exception as e:
            print(f"Error during poll: {e}")

        time.sleep(interval_seconds)

In [ ]:
poll_forever(interval_seconds=30)

## 5. Verification queries

In [13]:
pd.read_sql("SELECT COUNT(*) FROM flight_positions;", engine)

,count
0,306902


In [7]:
pd.read_sql(
    "SELECT * FROM flight_positions ORDER BY ingested_at DESC LIMIT 5;", engine
)

,id,icao24,callsign,origin_country,time_position,last_contact,longitude,latitude,baro_altitude,on_ground,velocity,true_track,vertical_rate,geo_altitude,squawk,spi,position_source,ingested_at
0,49609,aa9321,UAL915,United States,1786062878,1786062878,-62.1834,49.5611,11277.60,False,272.07,52.22,0.00,11871.96,2312,False,0,2026-08-07 00:34:42.347118
1,49610,801640,AIC2385,India,1786062877,1786062877,95.5716,10.8261,10363.20,False,235.20,279.44,0.00,NaN,2126,False,0,2026-08-07 00:34:42.347118
2,49607,e8027d,LPE2051,Chile,1786062878,1786062878,-79.8610,-6.6377,11277.60,False,230.59,152.06,0.33,12039.60,None,False,0,2026-08-07 00:34:42.347118
3,49608,c81bd2,ZKJMR,New Zealand,1786062877,1786062877,176.7774,-39.2334,1988.82,False,58.79,344.26,0.00,2118.36,None,False,0,2026-08-07 00:34:42.347118
4,49611,801645,AIC7JN,India,1786062878,1786062878,77.6974,13.5808,4968.24,False,171.73,15.29,8.13,5143.50,None,False,0,2026-08-07 00:34:42.347118
